# AgentSim-R — Run Simulation

This notebook runs the full simulation on Google Colab with GPU.

**Requirements:** A Colab runtime with GPU enabled (`Runtime → Change runtime type → T4 GPU`).

---

## Step 1: Clone repos, install dependencies, download model

In [ ]:
# 1a. Clone the simulation repo
REPO_URL = "https://github.com/gamercanned-arch/AgentSim-R.git"

!git clone {REPO_URL} /content/AgentSim-R
%cd /content/AgentSim-R
!pip install -q -r requirements.txt
print("✓ Simulation repo cloned and dependencies installed.")

In [ ]:
# 1b. Build llama.cpp with CUDA
!git clone https://github.com/ggerganov/llama.cpp /content/llama.cpp
%cd /content/llama.cpp
!cmake -B build -DGGML_CUDA=ON
!cmake --build build --config Release -j$(nproc)

# Verify the binary exists
!ls -lh build/bin/llama-server
print("✓ llama.cpp built successfully.")

# Return to project directory
%cd /content/AgentSim-R

In [ ]:
# ── 1b. Download Pre-compiled llama-server
import requests
import os

print("Fetching latest llama.cpp release info...")
release_info = requests.get("https://api.github.com/repos/ggerganov/llama.cpp/releases/latest").json()

# Find the Ubuntu CUDA 12 asset
download_url = None
for asset in release_info.get("assets", []):
    if "ubuntu" in asset["name"] and "cu12" in asset["name"]:
        download_url = asset["browser_download_url"]
        break

if not download_url:
    raise Exception("Could not find the pre-compiled CUDA binary!")

print(f"Downloading binary from: {download_url}")
!wget -q -O llama.zip {download_url}
!unzip -q -o llama.zip -d /content/llama-bin

# Make the server binary executable
!chmod +x /content/llama-bin/llama-server
print("✓ pre-compiled llama-server ready!")

In [ ]:
# ── 1c. Download the model ──
!pip install -q huggingface_hub
from huggingface_hub import hf_hub_download

hf_hub_download(
    repo_id="HauhauCS/Qwen3.5-4B-Uncensored-HauhauCS-Aggressive",
    filename="Qwen3.5-4B-Uncensored-HauhauCS-Aggressive-Q4_K_M.gguf",
    local_dir="/content/models/",
)

!ls -lh /content/models/*.gguf
print("✓ Model downloaded.")

---

## Step 2: Start llama-server

In [ ]:
import time, subprocess, requests
import os

MODEL_PATH = "/content/models/Qwen3.5-4B-Uncensored-HauhauCS-Aggressive-Q4_K_M.gguf"
CTX_SIZE   = 262144   

# Create the cache directory so server can write to it
os.makedirs("/content/AgentSim-R/cache", exist_ok=True)

# Start server with Jinja template and disk caching enabled
proc = subprocess.Popen(
    [
        "/content/llama-bin/llama-server",
        "--model", MODEL_PATH,
        "--n-gpu-layers", "999",
        "--ctx-size", str(CTX_SIZE),
        "--flash-attn",                     
        "--parallel", "6",                  
        "--slot-save-path", "/content/AgentSim-R/cache",  
        "--chat-template-file", "/content/AgentSim-R/prompts/template.jinja",
        "--host", "0.0.0.0",
        "--port", "8080",
        "--log-disable",
    ],
    stdout=open("/content/server.log", "w"),
    stderr=subprocess.STDOUT,
)
print(f"Server PID: {proc.pid}")

# Wait for server to become ready
for i in range(60):
    time.sleep(2)
    try:
        r = requests.get("http://localhost:8080/health", timeout=3)
        if r.status_code == 200:
            print(f"✓ Server ready after {(i+1)*2}s")
            break
    except requests.ConnectionError:
        pass
else:
    print("✗ Server did not start. Check /content/server.log:")
    !tail -n 40 /content/server.log

---

## Step 3: Run the simulation

In [ ]:
%cd /content/AgentSim-R
!python -m python.sim

---

## Step 4: Download logs

In [ ]:
import os
LOG_DIR = "/content/AgentSim-R/logs"

if os.path.isdir(LOG_DIR) and os.listdir(LOG_DIR):
    !zip -r /content/simulation_logs.zip {LOG_DIR}
    from google.colab import files
    files.download("/content/simulation_logs.zip")
    print(f"✓ Downloaded {len(os.listdir(LOG_DIR))} log files.")
else:
    print("✗ No logs found. Did the simulation run?")

---

## Cleanup (optional)

In [ ]:
# Kill the llama-server process
try:
    proc.terminate()
    proc.wait(timeout=5)
    print("✓ Server stopped.")
except Exception as e:
    print(f"Server cleanup: {e}")
    !pkill -f llama-server